In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================
# 1. ENVIRONMENT AND CONFIGURATION
# ============================================

import os
import random
import numpy as np
import torch

from google.colab import drive

# Mount Google Drive (skip if already mounted)
drive.mount('/content/drive')

# --------------------------------------------
# Project paths
# --------------------------------------------

DATASET_DIR = "/content/drive/MyDrive/segmentation_processed_256"

PROJECT_DIR = "/content/drive/MyDrive/BrainTumorSegmentation"

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# --------------------------------------------
# Reproducibility
# --------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------
# Device configuration
# --------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 50)
print("ENVIRONMENT CONFIGURATION")
print("=" * 50)

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

print("\nDataset directory:", DATASET_DIR)
print("Dataset exists:", os.path.exists(DATASET_DIR))

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)

print("\nSeed:", SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ENVIRONMENT CONFIGURATION
PyTorch version: 2.14.1+cpu
Device: cpu

Dataset directory: /content/drive/MyDrive/segmentation_processed_256
Dataset exists: True
Checkpoint directory: /content/drive/MyDrive/BrainTumorSegmentation/checkpoints
Results directory: /content/drive/MyDrive/BrainTumorSegmentation/results

Seed: 42


In [5]:
# Verify image-mask correspondence in the processed dataset

from pathlib import Path

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

for split in ["train", "test"]:
    image_dir = Path(DATASET_DIR) / split / "images"
    mask_dir = Path(DATASET_DIR) / split / "masks"

    images = sorted([f for f in image_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])
    masks = sorted([f for f in mask_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])

    image_stems = {f.stem for f in images}
    mask_stems = {f.stem for f in masks}

    missing_masks = image_stems - mask_stems
    missing_images = mask_stems - image_stems

    print(f"\n{'=' * 40}")
    print(f"{split.upper()} DATASET")
    print(f"{'=' * 40}")
    print(f"Images: {len(images)}")
    print(f"Masks: {len(masks)}")
    print(f"Matched pairs: {len(image_stems & mask_stems)}")
    print(f"Images without masks: {len(missing_masks)}")
    print(f"Masks without images: {len(missing_images)}")

    if missing_masks:
        print("Sample missing masks:", list(missing_masks)[:5])

    if missing_images:
        print("Sample missing images:", list(missing_images)[:5])


TRAIN DATASET
Images: 3933
Masks: 3933
Matched pairs: 3933
Images without masks: 0
Masks without images: 0

TEST DATASET
Images: 860
Masks: 860
Matched pairs: 860
Images without masks: 0
Masks without images: 0


In [6]:
# Copy the processed dataset from Google Drive to Colab local storage

import shutil
import os

LOCAL_DATASET_DIR = "/content/segmentation_processed_256"

if not os.path.exists(LOCAL_DATASET_DIR):
    shutil.copytree(DATASET_DIR, LOCAL_DATASET_DIR)
    print("Dataset copied successfully!")
else:
    print("Dataset already exists in local storage.")

print("Local dataset path:", LOCAL_DATASET_DIR)

Dataset copied successfully!
Local dataset path: /content/segmentation_processed_256


In [7]:
# Create PyTorch datasets and dataloaders for U-Net training

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import numpy as np
from pathlib import Path

class BrainTumorSegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.images = sorted(self.image_dir.glob("*"))
        self.masks = sorted(self.mask_dir.glob("*"))

        mask_map = {mask.stem: mask for mask in self.masks}
        self.pairs = [(img, mask_map[img.stem]) for img in self.images if img.stem in mask_map]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = np.array(Image.open(image_path).convert("L"), dtype=np.float32) / 255.0
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32) / 255.0
        mask = (mask >= 0.5).astype(np.float32)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask


train_dataset_full = BrainTumorSegmentationDataset(
    f"{LOCAL_DATASET_DIR}/train/images",
    f"{LOCAL_DATASET_DIR}/train/masks"
)

test_dataset = BrainTumorSegmentationDataset(
    f"{LOCAL_DATASET_DIR}/test/images",
    f"{LOCAL_DATASET_DIR}/test/masks"
)

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Testing batches: {len(test_loader)}")

Training samples: 3146
Validation samples: 787
Testing samples: 860
Training batches: 394
Validation batches: 99
Testing batches: 108


In [8]:
# Define the U-Net architecture for brain tumor segmentation

import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=64):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.encoder1 = DoubleConv(in_channels, base_channels)
        self.encoder2 = DoubleConv(base_channels, base_channels * 2)
        self.encoder3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.encoder4 = DoubleConv(base_channels * 4, base_channels * 8)

        self.bottleneck = DoubleConv(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(base_channels * 2, base_channels)

        self.output = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(self.pool(e2))
        e4 = self.encoder4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.decoder4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.decoder3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.decoder2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.decoder1(torch.cat([self.up1(d2), e1], dim=1))

        return self.output(d1)


model = UNet(in_channels=1, out_channels=1, base_channels=64).to(DEVICE)

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: U-Net")
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Device: {DEVICE}")

Model: U-Net
Total parameters: 31,036,481
Trainable parameters: 31,036,481
Device: cpu


In [9]:
# Test the U-Net forward pass and check GPU memory usage

images, masks = next(iter(train_loader))

images = images.to(DEVICE)
masks = masks.to(DEVICE)

with torch.no_grad():
    predictions = model(images)

print("Input shape:", images.shape)
print("Mask shape:", masks.shape)
print("Output shape:", predictions.shape)
print("Output device:", predictions.device)
print("GPU memory allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

assert predictions.shape == masks.shape, "Output and mask shapes do not match!"

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Input shape: torch.Size([8, 1, 256, 256])
Mask shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 1, 256, 256])
Output device: cpu
GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB


In [10]:
# Configure loss, optimizer, learning-rate scheduler, and mixed precision

import torch
import torch.nn as nn

class DiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probabilities = torch.sigmoid(logits)
        smooth = 1e-6

        intersection = (probabilities * targets).sum(dim=(1, 2, 3))
        denominator = probabilities.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))

        dice = (2 * intersection + smooth) / (denominator + smooth)
        dice_loss = 1 - dice.mean()

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


criterion = DiceBCELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

scaler = torch.amp.GradScaler("cuda")

EPOCHS = 40
EARLY_STOPPING_PATIENCE = 8

print("Loss: BCE + Dice")
print("Optimizer: AdamW")
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Scheduler: ReduceLROnPlateau")
print("Mixed precision: Enabled")
print("Maximum epochs:", EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)

Loss: BCE + Dice
Optimizer: AdamW
Initial learning rate: 0.0001
Scheduler: ReduceLROnPlateau
Mixed precision: Enabled
Maximum epochs: 40
Early stopping patience: 8


/tmp/ipykernel_1550/2295389527.py:44: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler("cuda")


In [11]:
# Implement the complete U-Net training and validation pipeline

import time
import copy
import numpy as np
from tqdm.auto import tqdm

def calculate_metrics(logits, targets, threshold=0.5):
    predictions = (torch.sigmoid(logits) >= threshold).float()

    smooth = 1e-6

    intersection = (predictions * targets).sum(dim=(1, 2, 3))
    pred_sum = predictions.sum(dim=(1, 2, 3))
    target_sum = targets.sum(dim=(1, 2, 3))

    dice = (2 * intersection + smooth) / (pred_sum + target_sum + smooth)
    iou = (intersection + smooth) / (pred_sum + target_sum - intersection + smooth)

    true_positive = (predictions * targets).sum()
    false_positive = (predictions * (1 - targets)).sum()
    false_negative = ((1 - predictions) * targets).sum()

    precision = (true_positive + smooth) / (true_positive + false_positive + smooth)
    recall = (true_positive + smooth) / (true_positive + false_negative + smooth)

    return dice.mean().item(), iou.mean().item(), precision.item(), recall.item()


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()

    running_loss = 0
    running_dice = 0
    running_iou = 0
    running_precision = 0
    running_recall = 0

    for images, masks in tqdm(loader, desc="Training", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        dice, iou, precision, recall = calculate_metrics(logits.detach(), masks)

        running_loss += loss.item()
        running_dice += dice
        running_iou += iou
        running_precision += precision
        running_recall += recall

    n = len(loader)

    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
        "precision": running_precision / n,
        "recall": running_recall / n
    }


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0
    running_dice = 0
    running_iou = 0
    running_precision = 0
    running_recall = 0

    for images, masks in tqdm(loader, desc="Validation", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, masks)

        dice, iou, precision, recall = calculate_metrics(logits, masks)

        running_loss += loss.item()
        running_dice += dice
        running_iou += iou
        running_precision += precision
        running_recall += recall

    n = len(loader)

    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
        "precision": running_precision / n,
        "recall": running_recall / n
    }


print("Training and validation functions are ready.")

Training and validation functions are ready.


In [12]:
# Train U-Net while saving the best validation Dice checkpoint

history = {
    "train_loss": [],
    "train_dice": [],
    "train_iou": [],
    "train_precision": [],
    "train_recall": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": [],
    "val_precision": [],
    "val_recall": [],
    "learning_rate": []
}

best_val_dice = 0.0
best_epoch = 0
epochs_without_improvement = 0

best_model_weights = None

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, DEVICE
    )

    val_metrics = validate_one_epoch(
        model, val_loader, criterion, DEVICE
    )

    scheduler.step(val_metrics["dice"])

    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - start_time

    for key in train_metrics:
        history[f"train_{key}"].append(train_metrics[key])
        history[f"val_{key}"].append(val_metrics[key])

    history["learning_rate"].append(current_lr)

    print(f"\nEpoch [{epoch}/{EPOCHS}] | Time: {epoch_time:.1f}s")
    print(f"Train Loss: {train_metrics['loss']:.4f} | Train Dice: {train_metrics['dice']:.4f}")
    print(f"Val Loss:   {val_metrics['loss']:.4f} | Val Dice:   {val_metrics['dice']:.4f}")
    print(f"Val IoU:    {val_metrics['iou']:.4f} | Precision:   {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f}")
    print(f"Learning Rate: {current_lr:.7f}")

    if val_metrics["dice"] > best_val_dice:
        best_val_dice = val_metrics["dice"]
        best_epoch = epoch
        epochs_without_improvement = 0
        best_model_weights = copy.deepcopy(model.state_dict())

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_dice": best_val_dice,
            "val_metrics": val_metrics,
            "history": history
        }, f"{CHECKPOINT_DIR}/best_unet.pth")

        print(f"Best model saved! Validation Dice: {best_val_dice:.4f}")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)
print(f"Best epoch: {best_epoch}")
print(f"Best validation Dice: {best_val_dice:.4f}")
print(f"Checkpoint: {CHECKPOINT_DIR}/best_unet.pth")

Training:   0%|          | 0/394 [00:00<?, ?it/s]

Validation:   0%|          | 0/99 [00:00<?, ?it/s]


Epoch [1/40] | Time: 103.1s
Train Loss: 0.6051 | Train Dice: 0.4684
Val Loss:   0.5400 | Val Dice:   0.6334
Val IoU:    0.5410 | Precision:   0.7839 | Recall: 0.6375
Learning Rate: 0.0001000
Best model saved! Validation Dice: 0.6334


Training:   0%|          | 0/394 [00:00<?, ?it/s]

Validation:   0%|          | 0/99 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d1ede9dcc20>
Traceback (most recent call last):
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7d1ede9dcc20>    Traceback (most recent call last):

self._shutdown_workers()
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d1ede9dcc20>    <function _MultiProcessingDataLoaderIter.__del__ at 0x7d1ede9dcc20>    if w.is_alive():

self._shutdown_workers()Traceback (most recent call last):

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__


... (130 outputs left out)

Training:   0%|          | 0/394 [00:00<?, ?it/s]

self._shutdown_workers()
Exception ignored in:   File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7d1ede9dcc20>if w.is_alive():

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1706, in __del__
        assert self._parent_pid == os.getpid(), 'can only test a child process'self._shutdown_workers()

AssertionError  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1675, in _shutdown_workers
:     can only test a child processif w.is_alive():

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d1ede9dcc20>    
Traceback (most recent call last):
assert self._parent_pid == os.getpid(), '

Validation:   0%|          | 0/99 [00:00<?, ?it/s]


Epoch [34/40] | Time: 108.0s
Train Loss: 0.0444 | Train Dice: 0.9244
Val Loss:   0.0877 | Val Dice:   0.8492
Val IoU:    0.7792 | Precision:   0.9050 | Recall: 0.8494
Learning Rate: 0.0000063

Early stopping triggered at epoch 34.

TRAINING COMPLETED
Best epoch: 26
Best validation Dice: 0.8493
Checkpoint: /content/drive/MyDrive/BrainTumorSegmentation/checkpoints/best_unet.pth
